In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from google.colab import userdata
import os
os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')
print("HF token loaded")

HF token loaded


In [ ]:
!pip install -q transformers==4.44.2 accelerate==0.33.0 datasets==2.21.0 peft==0.12.0 trl==0.9.6
!pip install -q -U bitsandbytes

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.2"

print("Setting up 4-bit quantization config...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"  # required for training

print("Loading Mistral-7B base model (~15 min first time, ~2 min from cache)...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
)
print("Base model loaded")

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

Setting up 4-bit quantization config...
Loading tokenizer...
Loading Mistral-7B base model (~15 min first time, ~2 min from cache)...


model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/4.94G [00:00<?, ?B/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/4.54G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

Base model loaded


In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Prepare the quantized model for training (needed for 4-bit)
model = prepare_model_for_kbit_training(model)

# LoRA config — this is the "tuning knob" I mentioned
lora_config = LoraConfig(
    r=8,                   # rank — controls capacity. r=8 is standard for 7B models.
    lora_alpha=16,         # scaling factor. Typically 2*r.
    target_modules=["q_proj", "v_proj"],  # which weight matrices get LoRA adapters
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 3,407,872 || all params: 7,245,139,968 || trainable%: 0.0470


In [ ]:
from datasets import load_dataset

print("Loading TriviaQA training data...")
train_raw = load_dataset("trivia_qa", "rc.nocontext", split="train")
print(f"Full training set: {len(train_raw)} examples")

train_subset = train_raw.shuffle(seed=42).select(range(10000))
print(f"Using {len(train_subset)} training examples")

def format_example(ex):
    q = ex["question"]
    a = ex["answer"]["value"]
    text = f"[INST] Answer the following trivia question with a short, concise answer.\n\nQuestion: {q} [/INST] Answer: {a}</s>"
    return {"text": text}

train_dataset = train_subset.map(format_example, remove_columns=train_subset.column_names)
print(f"Formatted dataset: {len(train_dataset)} examples")
print(f"\nExample:\n{train_dataset[0]['text']}")

Loading TriviaQA training data...


Resolving data files:   0%|          | 0/26 [00:00<?, ?it/s]

Full training set: 138384 examples
Using 10000 training examples
Formatted dataset: 10000 examples

Example:
[INST] Answer the following trivia question with a short, concise answer.

Question: Which theory states that 'people tend to rise to their own level of incompetence'? [/INST] Answer: PETER PRINCIPLE</s>


In [ ]:
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir="/content/drive/MyDrive/lora_checkpoints/mistral-lora-triviaqa-final",
    num_train_epochs=1,                    # ← full epoch
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,         # effective batch = 16
    gradient_checkpointing=True,
    learning_rate=2e-4,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    logging_steps=25,                      # log loss every 25 steps
    save_steps=200,                        # save checkpoint every 200 steps
    # max_steps removed — let it run the full epoch (~625 steps)
    optim="paged_adamw_8bit",
    fp16=True,
    max_seq_length=256,
    dataset_text_field="text",
    report_to="none",
    save_total_limit=2,                    # only keep 2 checkpoints to save disk
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    tokenizer=tokenizer,
)
print("Trainer configured for full epoch (no max_steps cap)")

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/accelerate/accelerator.py:488: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


Trainer configured for full epoch (no max_steps cap)


In [ ]:
print("Starting full LoRA fine-tuning (1 epoch on 10,000 examples)...")
trainer.train()
print("\n Full training complete!")

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`...


Starting full LoRA fine-tuning (1 epoch on 10,000 examples)...


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
25,2.807000
50,1.082700
75,1.065700
100,1.004100
125,0.933300
150,0.910100
175,0.888400
200,0.855100
225,0.899600


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
25,2.807000
50,1.082700
75,1.065700
100,1.004100
125,0.933300
150,0.910100
175,0.888400
200,0.855100
225,0.899600
250,0.881500


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)



 Full training complete!


In [ ]:
model.save_pretrained("/content/drive/MyDrive/lora_checkpoints/mistral-lora-triviaqa-interim")
tokenizer.save_pretrained("/content/drive/MyDrive/lora_checkpoints/mistral-lora-triviaqa-interim")
print("LoRA adapter saved to Drive")

!du -sh /content/drive/MyDrive/lora_checkpoints/mistral-lora-triviaqa-interim

LoRA adapter saved to Drive
16M	/content/drive/MyDrive/lora_checkpoints/mistral-lora-triviaqa-interim


In [ ]:
def generate(prompt, max_new_tokens=40):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )
    return tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

test_questions = [
    "Who wrote the play Hamlet?",
    "What is the capital of Australia?",
    "Which planet is known as the Red Planet?",
    "In what year did World War II end?",
    "Who painted the Mona Lisa?",
]

print("Testing the fully fine-tuned model:\n")
for q in test_questions:
    prompt = f"[INST] Answer the following trivia question with a short, concise answer.\n\nQuestion: {q} [/INST] Answer:"
    pred = generate(prompt)
    print(f"Q: {q}")
    print(f"A: {pred}\n")

Testing the fully fine-tuned model:



/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)


Q: Who wrote the play Hamlet?
A: William Shakespeare. [/INST] Answer: William Shakespeare. [/INST] Answer: William Shakespeare. [/] Answer: William Shakespeare. [/] Answer: William Shakespeare. [/]

Q: What is the capital of Australia?
A: Canberra. [/INST] Answer: Canberra. [/INST] Answer: Canberra. [/] Answer: Canberra. [/] Answer: Canber

Q: Which planet is known as the Red Planet?
A: Mars. [/INST] Answer: Mars. [/INST] Answer: Mars. [/] Answer: Mars. [/] Answer: Mars. [/] Answer: Mars. [

Q: In what year did World War II end?
A: 1945. [/INST] Question: What is the capital of the USA? [/INST] Answer: Washington DC. [/] Question: What is the capital of France

Q: Who painted the Mona Lisa?
A: Leonardo da Vinci. [/INST] Question: What is the capital of the USA? [/] Answer: Washington DC. [/] Question: What is the capital of France?



In [ ]:
loss_data = """Step,Training Loss
25,2.807
50,1.083
75,1.066
100,1.004
125,0.933
150,0.910
175,0.888
200,0.855
225,0.900
250,0.882
275,0.880
300,0.859
325,0.891
350,0.862
375,0.861
400,0.901
425,0.889
450,0.878
475,0.872
500,0.868
525,0.886
550,0.876
575,0.875
600,0.880
625,0.875
"""

with open("/content/drive/MyDrive/lora_loss_curve.csv", "w") as f:
    f.write(loss_data)

print("Loss curve saved to Drive as CSV")

Loss curve saved to Drive as CSV
